In [1]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


项目根目录: /Users/lizhipeng/个人成长/职业发展/求职面试/项目/RetailInsight


# 05 · 流失预测模型
> 目标：给每个会员一个流失概率，并给出可运营的名单与阈值依据。
> **防泄漏是底线**：特征只用观察窗，标签只看标签窗。

In [2]:
import json

import pandas as pd

from retailinsight.io.loader import read_parquet
from retailinsight.settings import MART_CHURN, REPORTS_DIR

metrics = json.loads((REPORTS_DIR / "metrics_churn.json").read_text(encoding="utf-8"))
print("backend:", metrics["backend"])
print("时间切分:", metrics["split"])
print("特征数:", metrics["n_features"])
pd.DataFrame({"LR 基线": metrics["lr_baseline"], "GBDT 主模型": metrics["gbdt"],
              "RandomForest": metrics["random_forest"]})


backend: lightgbm
时间切分: {'train': ['2021-04-01', '2021-07-01', '2021-10-01'], 'valid': ['2022-01-01'], 'test': ['2022-04-01']}
特征数: 31


,LR 基线,GBDT 主模型,RandomForest
auc,0.7995,0.8179,0.8300
ks,0.4521,0.4836,0.4936
lift_top20,2.3000,2.3500,2.3400
capture_top20,0.4592,0.4694,0.4677
base_rate,0.3672,0.3672,0.3672


## 5.1 验收线核对

In [3]:
gb, lr = metrics["gbdt"], metrics["lr_baseline"]
upper = 1 / gb["base_rate"]
print(f"AUC 增益: {gb['auc'] - lr['auc']:+.4f}（验收线 +0.03）→ "
      f"{'达标' if gb['auc'] - lr['auc'] >= 0.03 else '未达标'}")
print(f"KS: {gb['ks']:.4f}（验收线 ≥0.30）→ {'达标' if gb['ks'] >= 0.30 else '未达标'}")
print(f"Lift@20%: {gb['lift_top20']:.2f} / 理论上限 {upper:.2f} = "
      f"{gb['lift_top20'] / upper:.1%}（验收线 ≥85%）")


AUC 增益: +0.0184（验收线 +0.03）→ 未达标
KS: 0.4836（验收线 ≥0.30）→ 达标
Lift@20%: 2.35 / 理论上限 2.72 = 86.3%（验收线 ≥85%）


## 5.2 特征重要性（SHAP 或 permutation importance 降级）

In [4]:
imp = pd.Series({k: v for k, v in metrics["feature_importance"].items() if k != "method"},
                 dtype="float64").sort_values(ascending=False)
imp.head(15)


items_per_txn           0.0471
brand_entropy           0.0344
recency_days            0.0335
gap_mean                0.0233
store_cnt               0.0221
frequency               0.0211
trend_90d               0.0210
freq_90d                0.0199
n_txn                   0.0182
days_since_2nd_last     0.0171
tenure_days             0.0161
seasonality_strength    0.0141
n_categories            0.0139
amt_90d                 0.0131
amt_30d                 0.0098
dtype: float64

## 5.3 验证集上的阈值调优
> 按业务成本最优（漏判 : 误触 = 8 : 1），与《决策建议报告》S2 同口径。

In [5]:
th = metrics["thresholds"]
print(json.dumps(th, ensure_ascii=False, indent=2))


{
  "action": 0.28,
  "mid": 0.4700000000000001,
  "high": 0.5800000000000001,
  "cost_fn": 8.0,
  "cost_fp": 1.0,
  "n_valid": 20613,
  "base_rate": 0.2945
}


## 5.4 名单产出与分层

In [6]:
churn = read_parquet(MART_CHURN)
print(f"打分覆盖 {len(churn):,} 人")
print(churn["risk_level"].value_counts())
print(f"\nTop20 高危名单平均流失概率: {churn.nlargest(int(len(churn)*0.2), 'churn_prob')['churn_prob'].mean():.4f}")
print("\n单会员归因字段:", [c for c in churn.columns if 'shap' in c.lower()])


打分覆盖 22,777 人
risk_level
低    15864
高     5014
中     1899
Name: count, dtype: int64

Top20 高危名单平均流失概率: 0.8447

单会员归因字段: ['top_shap_feature']


## 5.5 概率分布

In [7]:
from retailinsight.viz import charts

print(charts.churn_hist(churn["churn_prob"].to_numpy()))
print(charts.lift_curve(churn["churn"].to_numpy(), churn["churn_prob"].to_numpy()))


/Users/lizhipeng/个人成长/职业发展/求职面试/项目/RetailInsight/reports/figures/08_churn_hist.png
/Users/lizhipeng/个人成长/职业发展/求职面试/项目/RetailInsight/reports/figures/07_lift_curve.png
